In [1]:
import geemap
import ee

import random
from shapely.geometry import Polygon, mapping
import geopandas as gpd

In [2]:
m = geemap.Map()
m

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [5]:
roi = m.user_roi

In [16]:
aoi

ee.Geometry({
  "functionInvocationValue": {
    "functionName": "GeometryConstructors.Polygon",
    "arguments": {
      "coordinates": {
        "constantValue": [
          [
            [
              -97.021749,
              31.365537
            ],
            [
              -97.021749,
              31.653679
            ],
            [
              -96.581087,
              31.653679
            ],
            [
              -96.581087,
              31.365537
            ]
          ]
        ]
      },
      "evenOdd": {
        "constantValue": true
      }
    }
  }
})

In [7]:
aoi = ee.Geometry.Rectangle([-97.021749, 31.653679, -96.581087, 31.365537])
roi = ee.Geometry.BBox(-97.528381, 31.781736, -96.630249, 31.292487)

In [13]:
all_panels = geemap.shp_to_ee('lulc/old_training_data/all_solar.shp')
big_fm162 = geemap.shp_to_ee("Big_creek/big_creek_downstream.shp")
brushy_down = geemap.shp_to_ee('Brushy_Creek/brushy_downstream.shp')
hugpen_down = geemap.shp_to_ee('Hogpen_creek/hugpen_downstream.shp')

style = {'color': 'ffffff', 'fillColor': '00000000'}

In [14]:
m.addLayer(all_panels.style(**style), {}, 'solar')
m.addLayer(aoi, {'color': 'orange'}, 'aoi')
#m.addLayer(roi, {'color': 'green'}, 'roi')
m.addLayer(big_fm162, {'color': 'yellow'}, 'big162')
m.addLayer(brushy_down, {'color': 'green'}, 'brushy')
m.addLayer(hugpen_down, {'color': 'blue'}, 'hugpen')
m.centerObject(roi)

In [15]:
# Date range
start_date = '2023-10-01'
end_date = '2023-11-01'


# Load Sentinel-2 Image Collection
collection = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterBounds(aoi) \
    .filterDate(start_date, end_date) \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 5))#.map(maskS2clouds)  # Filter <0.1% cloud cover


collection.size().getInfo()

16

In [17]:
image = collection.median().clip(aoi)

# Select spectral bands
bands = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']  # Blue, Green, Red, NIR, SWIR1, SWIR2

# Calculate NDVI
ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')

# Calculate NDBI
ndbi = image.normalizedDifference(['B11', 'B8']).rename('NDBI')

# Calculate MNDWI
mndwi = image.normalizedDifference(['B3', 'B11']).rename('MNDWI')

# Combine bands and indices into a single image
features_image = image.select(bands).addBands([ndvi, ndbi, mndwi])

# List of features (bands + indices)
features = bands + ['NDVI', 'NDBI', 'MNDWI']

In [18]:
visualization = {
    'min': 0.0,
    'max': 3000,
    'bands': ['B4', 'B3', 'B2'],
}

# Visualize the classified image
Map = geemap.Map()
Map.centerObject(aoi)  # Set the center to your region of interest
Map.addLayer(image, visualization, "Sentinel")
# Map.addLayer(aoi, {'color': 'orange'}, 'aoi')
# Map.addLayer(all_panels.style(**style), {}, 'solar')
Map.addLayer(big_fm162, {'color': 'yellow'}, 'big162')
Map.addLayer(brushy_down, {'color': 'yellow'}, 'brushy_down')
Map.addLayer(hugpen_down, {'color': 'yellow'}, 'hugpen_down')
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [19]:
bareland_shp = 'lulc/taining_data_modified/bareland_points_newer.shp'
building_shp = 'lulc/taining_data_modified/buildingPoints_new.shp'
grass_shp = 'lulc/taining_data_modified/grass_points_newer.shp'
solar_shp = 'lulc/taining_data_modified/solar_points_new.shp'
denseVeg_shp = 'lulc/taining_data_modified/vegnew_veg_dense_points_new.shp'
water_shp = 'lulc/taining_data_modified/fixed_water_points_new.shp'


bareland = geemap.shp_to_ee(bareland_shp)
building = geemap.shp_to_ee(building_shp)
grass = geemap.shp_to_ee(grass_shp)
solar = geemap.shp_to_ee(solar_shp)
denseVeg = geemap.shp_to_ee(denseVeg_shp)
water = geemap.shp_to_ee(water_shp)

In [20]:
Map = geemap.Map()
Map.addLayer(image, visualization, "Sentinel")
Map.addLayer(bareland, {'color': 'saddlebrown'}, 'Bare Land')
Map.addLayer(building, {'color': 'red'}, 'Building')
Map.addLayer(grass, {'color': 'lightgreen'}, 'Grassland')
Map.addLayer(solar, {'color': 'gray'}, 'Solar Panels')
Map.addLayer(denseVeg, {'color': 'darkgreen'}, 'Dense Vegetation')
Map.addLayer(water, {'color': 'blue'}, 'Water')

# Center map on all layers
Map.centerObject(aoi, zoom=12)

# Display map
Map

Map(center=[31.509722736977572, -96.80141799999997], controls=(WidgetControl(options=['position', 'transparent…

In [21]:
# Combine them into one feature collection with a class property
solar = solar.set('class', 1)
building = building.set('class', 2)
bareland = bareland.set('class', 3)
water = water.set('class', 4)
denseVeg = denseVeg.set('class', 6)
grass = grass.set('class', 5)


# Merge them into a single collection
training_data = solar.merge(building).merge(bareland).merge(water).merge(denseVeg).merge(grass)

In [23]:
# Sample the image using the training data
training_samples = features_image.sampleRegions(
    collection=training_data,
    properties=['class'],
    scale=10
)


###############################################

# Assume 'sample' is your FeatureCollection of labeled points
sample = training_samples.randomColumn()

# 75% training, 25% validation
training = sample.filter(ee.Filter.lt('random', 0.75))
validation = sample.filter(ee.Filter.gte('random', 0.75))


print('Original sample size:', sample.size().getInfo())
print('Training size:', training.size().getInfo())
print('Validation size before filtering:', validation.size().getInfo())

###############################################


# Train the classifier
classifier = ee.Classifier.smileRandomForest(numberOfTrees=100).train(
    features=training,
    classProperty='class',
    inputProperties=features
)


# Apply the classifier
classified_image = features_image.classify(classifier)

Original sample size: 24218
Training size: 18113
Validation size before filtering: 6105


In [25]:
# Create a color palette for each class (solar, bareland, water, denseVeg, grass, building)
palette = ['yellow', 'green', 'red', 'blue', 'orange', 'purple']  # Choose colors for each class


visualization = {
    'min': 0.0,
    'max': 3000,
    'bands': ['B4', 'B3', 'B2'],
}

# Visualize the classified image
Map = geemap.Map()
Map.centerObject(aoi)  # Set the center to your region of interest
Map.addLayer(image, visualization, "Sentinel")
Map.addLayer(classified_image, {'min': 1, 'max': 6, 'palette': palette}, 'Classified Image')

Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [26]:
# Filter out null values
validation = validation.filter(ee.Filter.notNull(bands))

# Classify the filtered validation set
validated = validation.classify(classifier)

# Compute confusion matrix: actual ('class') vs predicted ('classification')
confMatrix = validated.errorMatrix('class', 'classification')


# Print metrics
print('Confusion Matrix:', confMatrix.getInfo())
print('Overall Accuracy:', confMatrix.accuracy().getInfo())
print('Kappa:', confMatrix.kappa().getInfo())
print('Recall (Producer\'s Accuracy):', confMatrix.producersAccuracy().getInfo())
print('Precision (User\'s Accuracy):', confMatrix.consumersAccuracy().getInfo())

Confusion Matrix: [[0, 0, 0, 0, 0, 0, 0], [0, 1445, 6, 7, 0, 22, 1], [0, 14, 230, 1, 0, 4, 3], [0, 20, 6, 1425, 0, 6, 0], [0, 1, 3, 0, 143, 1, 0], [0, 40, 5, 27, 4, 2086, 0], [0, 4, 1, 0, 1, 3, 596]]
Overall Accuracy: 0.9705159705159705
Kappa: 0.9605433817959034
Recall (Producer's Accuracy): [[0], [0.9756920999324781], [0.9126984126984127], [0.9780370624571036], [0.9662162162162162], [0.9648473635522664], [0.9851239669421488]]
Precision (User's Accuracy): [[0, 0.9481627296587927, 0.9163346613545816, 0.976027397260274, 0.9662162162162162, 0.9830348727615457, 0.9933333333333333]]


In [27]:
nir_band = image.select('B8')

nir_gray = nir_band.multiply(255).divide(ee.Number(nir_band.reduceRegion(
    reducer=ee.Reducer.max(),
    geometry=aoi,
    scale=10,
    maxPixels=1e8  # Increase maxPixels (adjust as needed)
).get('B8'))).toUint8()


glcm_texture = nir_gray.glcmTexture(size=3)

# Select key GLCM texture variables using the correct band names
texture_variables = glcm_texture.select([
    'B8_asm',      # Angular Second Moment
    'B8_contrast', # Contrast
    'B8_corr',     # Correlation
    'B8_ent',      # Entropy
    'B8_var',      # Variance
    'B8_idm',      # Inverse Difference Moment
    'B8_svar',     # Sum Variance
    'B8_diss'      # Dissimilarity
])

In [28]:
#spectral_bands = image.select(bands)

#combined_image = features_image.addBands([texture_variables, dem, slope])
combined_image = features_image.addBands(texture_variables)


# Sample the combined image
training_samples = combined_image.sampleRegions(
    collection=training_data,
    properties=['class'],
    scale=10
)


###############################################

# Sample is the FeatureCollection of labeled points
sample = training_samples.randomColumn()

# 75% training, 25% validation
training = sample.filter(ee.Filter.lt('random', 0.75))
validation = sample.filter(ee.Filter.gte('random', 0.75))


print('Original sample size:', sample.size().getInfo())
print('Training size:', training.size().getInfo())
print('Validation size before filtering:', validation.size().getInfo())

###############################################


# Train the classifier
classifier = ee.Classifier.smileRandomForest(numberOfTrees=100).train(
    features=training,
    classProperty='class',
    inputProperties=combined_image.bandNames()
)


# Apply the classifier
classified_image = combined_image.classify(classifier)


visualization = {
    'min': 0.0,
    'max': 3000,
    'bands': ['B4', 'B3', 'B2'],
}

# Visualize the classified image
Map = geemap.Map()
Map.centerObject(aoi)  # Set the center to your region of interest
Map.addLayer(image, visualization, "Sentinel")


# Add classification to the map
# Create a color palette for each class (solar, bareland, water, denseVeg, grass, building)
palette = ['red', 'yellow', 'blue', 'green', 'orange', 'purple']  # Choose colors for each class

# Visualize the classified image
#Map = geemap.Map(center=[37.7749, -122.4194], zoom=10)  # Set the center to your region of interest
Map.addLayer(classified_image, {'min': 1, 'max': 6, 'palette': palette}, 'Classified Image')


# Display the map
Map

Original sample size: 24218
Training size: 18192
Validation size before filtering: 6026


Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [45]:
# Filter out null values
validation = validation.filter(ee.Filter.notNull(bands))

# Classify the filtered validation set
validated = validation.classify(classifier)

# Compute confusion matrix: actual ('class') vs predicted ('classification')
confMatrix = validated.errorMatrix('class', 'classification')

In [46]:
# Print metrics
print('Confusion Matrix:', confMatrix.getInfo())
print('Overall Accuracy:', confMatrix.accuracy().getInfo())
print('Kappa:', confMatrix.kappa().getInfo())
print('Recall (Producer\'s Accuracy):', confMatrix.producersAccuracy().getInfo())
print('Precision (User\'s Accuracy):', confMatrix.consumersAccuracy().getInfo())

Confusion Matrix: [[0, 0, 0, 0, 0, 0, 0], [0, 1492, 4, 6, 0, 8, 0], [0, 2, 251, 4, 0, 2, 2], [0, 20, 3, 1385, 0, 12, 0], [0, 0, 0, 0, 160, 0, 1], [0, 50, 7, 42, 3, 2112, 1], [0, 4, 1, 0, 1, 5, 584]]
Overall Accuracy: 0.9711132749107433
Kappa: 0.9613496847554989
Recall (Producer's Accuracy): [[0], [0.9880794701986755], [0.9616858237547893], [0.9753521126760564], [0.9937888198757764], [0.9534988713318284], [0.9815126050420168]]
Precision (User's Accuracy): [[0, 0.951530612244898, 0.943609022556391, 0.9638135003479471, 0.975609756097561, 0.9873772791023843, 0.9931972789115646]]


In [18]:
import numpy as np

# 1. Get fresh values from the confusion matrix
recall_raw = confMatrix.producersAccuracy().getInfo()
precision_raw = confMatrix.consumersAccuracy().getInfo()

print("Raw recall   :", recall_raw)
print("Raw precision:", precision_raw)

# 2. Convert to 1D numeric arrays

# recall_raw is column-like: [[0], [0.97], ...]
recall = np.array([row[0] for row in recall_raw], dtype=float)

# precision_raw is row-like: [[0, 0.95, 0.91, ...]]
precision = np.array(precision_raw[0], dtype=float)

print("Flattened recall   :", recall.tolist())
print("Flattened precision:", precision.tolist())

# 3. Compute F1
den = precision + recall
f1 = np.where(den > 0, 2 * precision * recall / den, 0)

print("F1-scores:", f1.tolist())

Raw recall   : [[0], [0.9756920999324781], [0.9126984126984127], [0.9780370624571036], [0.9662162162162162], [0.9648473635522664], [0.9851239669421488]]
Raw precision: [[0, 0.9481627296587927, 0.9163346613545816, 0.976027397260274, 0.9662162162162162, 0.9830348727615457, 0.9933333333333333]]
Flattened recall   : [0.0, 0.9756920999324781, 0.9126984126984127, 0.9780370624571036, 0.9662162162162162, 0.9648473635522664, 0.9851239669421488]
Flattened precision: [0.0, 0.9481627296587927, 0.9163346613545816, 0.976027397260274, 0.9662162162162162, 0.9830348727615457, 0.9933333333333333]
F1-scores: [0.0, 0.961730449251248, 0.9145129224652087, 0.977031196434693, 0.9662162162162162, 0.9738562091503269, 0.9892116182572614]


In [26]:
# Apply a mode filter (majority filter) with a kernel size of 3x3
classified_smoothed = classified_image.focal_mode(radius=1, units='pixels')

vis_params = {
  'min': 0,
  'max': 6,  # Adjust based on the range of your classification values
  'palette': ['red', 'black', 'white', 'orange', 'lightblue', 'lightgreen', 'darkgreen'] # Add appropriate colors for classes
}


# Visualize the smoothed classification
Map.addLayer(classified_smoothed, vis_params, 'Smoothed Classification')

In [ ]:
#geemap.ee_export_vector(large_polygon, filename='sample_points/large_polygon.shp') 

In [27]:
# Step 1: Load the shapefile containing the solar panel siting areas
non_siting = geemap.shp_to_ee('lulc/old_training_data/aoi_polygon.shp')

# Step 3: Mask the raster with the polygon (clip the raster to the polygon)
masked_raster = classified_smoothed.clip(non_siting)

# Step 4: Reclassify the values inside the polygon (e.g., change values of 1 to 3)
# Here we use .where() to change value 1 to 3 inside the polygon
corrected_classified_image = classified_smoothed.where(masked_raster.eq(1), 3)


# Visualization parameters
vis_params = {
    'min': 1,  # Adjust according to your class values
    'max': 6, 
    'palette': ['red', 'yellow', 'orange', 'blue', 'lightgreen', 'darkgreen'],
}

# Visualize the original and corrected classifications
#Map.addLayer(image, visualization, "Sentinel")
Map.addLayer(corrected_classified_image, vis_params, 'Corrected Classification (Solar Panel Masked)')
Map.addLayer(big_fm162, {'color': 'orange'}, 'big162')

In [31]:
# Date range
start_date = '2024-10-01'
end_date = '2024-11-01'


# Load Sentinel-2 Image Collection
collection = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterBounds(aoi) \
    .filterDate(start_date, end_date) \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 5))#.map(maskS2clouds)  # Filter <0.1% cloud cover


image = collection.median().clip(aoi)

# Select spectral bands
bands = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']  # Blue, Green, Red, NIR, SWIR1, SWIR2

# Calculate NDVI
ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')

# Calculate NDBI
ndbi = image.normalizedDifference(['B11', 'B8']).rename('NDBI')

# Calculate MNDWI
mndwi = image.normalizedDifference(['B3', 'B11']).rename('MNDWI')

# Combine bands and indices into a single image
features_image = image.select(bands).addBands([ndvi, ndbi, mndwi])

# List of features (bands + indices)
features = bands + ['NDVI', 'NDBI', 'MNDWI']



nir_band = image.select('B8')

nir_gray = nir_band.multiply(255).divide(ee.Number(nir_band.reduceRegion(
    reducer=ee.Reducer.max(),
    geometry=aoi,
    scale=10,
    maxPixels=1e8  # Increase maxPixels (adjust as needed)
).get('B8'))).toUint8()


glcm_texture = nir_gray.glcmTexture(size=3)

# Select key GLCM texture variables using the correct band names
texture_variables = glcm_texture.select([
    'B8_asm',      # Angular Second Moment
    'B8_contrast', # Contrast
    'B8_corr',     # Correlation
    'B8_ent',      # Entropy
    'B8_var',      # Variance
    'B8_idm',      # Inverse Difference Moment
    'B8_svar',     # Sum Variance
    'B8_diss'      # Dissimilarity
])

combined_image = features_image.addBands(texture_variables)


# Apply the classifier
classified_image = combined_image.classify(classifier)

visualization = {
    'min': 0.0,
    'max': 3000,
    'bands': ['B4', 'B3', 'B2'],
}

# Visualize the classified image
Map = geemap.Map()
Map.centerObject(aoi)  # Set the center to your region of interest
Map.addLayer(image, visualization, "Sentinel")


# Add classification to the map
# Create a color palette for each class (solar, bareland, water, denseVeg, grass, building)
palette = ['red', 'yellow', 'blue', 'green', 'orange', 'purple']  # Choose colors for each class

# Visualize the classified image
#Map = geemap.Map(center=[37.7749, -122.4194], zoom=10)  # Set the center to your region of interest
Map.addLayer(classified_image, {'min': 1, 'max': 6, 'palette': palette}, 'Classified Image')
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…